# Car damage detection — retrain on VehiDE + an undamaged-car dataset

Runs on a **Kaggle GPU notebook** (Settings → Accelerator → GPU T4 x2 or P100) or **Google Colab**.

**Attach these datasets** (Kaggle: *Add Input*):
1. **VehiDE** — `hendrichscullen/vehide-dataset-automatic-vehicle-damage-detection` (damaged cars, polygon masks, 7 classes)
2. **Undamaged cars** for the classifier, e.g. `vinayjose/car-damage-dataset` (its `01-whole` folders give undamaged cars, its `00-damage` folders give whole-car damaged photos) and/or a Stanford Cars mirror.

VehiDE has only damaged cars, so (2) is what teaches the ResNet-50 what an *undamaged* car looks like.
The YOLO detector is trained and evaluated on VehiDE alone, using VehiDE's **official validation split**.

Outputs land in `/kaggle/working/export/`: `resnet50_damage.pth`, `yolo26_damage.pt`, `metrics.md`, plots.

In [ ]:
REPO_URL = "https://github.com/kb2717/DC_Manufacturing.git"
BRANCH = "prefinal"

# Training profile. A T4 manages roughly 12-15 min/epoch at 640px on VehiDE;
# Kaggle sessions stop at 12 h, so "fast" fits one session. "full" reproduces the
# original 1280px / 150-epoch run and needs an A100-class GPU or several resumed sessions.
PROFILE = "fast"            # "fast" | "full"
YOLO_START = "yolo26s-seg.pt"   # or "yolo26m-seg.pt" (bigger, slower, a bit more accurate)

# Where attached datasets live. Kaggle mounts them under /kaggle/input.
# On Colab, download them (e.g. with the kaggle CLI) into /content/data and change this.
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"

In [ ]:
import os, subprocess
if not os.path.exists(f"{WORK}/DC_Manufacturing"):
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO_URL, f"{WORK}/DC_Manufacturing"], check=True)
%cd {WORK}/DC_Manufacturing/dc_project/dc_proj/Backend
!pip install -q -U ultralytics gradio openai python-dotenv
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Find VehiDE and the undamaged images

In [ ]:
import json, os
from collections import Counter

IMG_EXTS = (".jpg", ".jpeg", ".png")

def vehide_jsons(root):
    # VIA-style annotation files: a dict of image entries that carry 'regions'.
    found = []
    for dp, _, fs in os.walk(root):
        for f in fs:
            if f.lower().endswith(".json") and os.path.getsize(os.path.join(dp, f)) > 100:
                p = os.path.join(dp, f)
                try:
                    d = json.load(open(p))
                except Exception:
                    continue
                if isinstance(d, dict) and any(isinstance(v, dict) and "regions" in v for v in list(d.values())[:20]):
                    found.append((p, d))
    return found

def best_image_dir(root, names):
    # The folder that holds the most of the JSON's image files.
    hits = Counter()
    for dp, _, fs in os.walk(root):
        hits[dp] = sum(1 for f in fs if f in names)
    return hits.most_common(1)[0]

jsons = vehide_jsons(INPUT_ROOT)
assert jsons, "No VehiDE annotation JSON found — is the VehiDE dataset attached?"
split = {}
for p, d in jsons:
    names = {os.path.basename(v.get("name", v.get("filename", k))) for k, v in d.items() if isinstance(v, dict)}
    img_dir, n = best_image_dir(INPUT_ROOT, names)
    kind = "val" if "val" in os.path.basename(p).lower() else "train"
    print(f"{kind:5s} {p}  -> {img_dir}  ({n}/{len(names)} images found)")
    split[kind] = (p, img_dir)
assert "train" in split, "Could not tell which JSON is the training split."

# Undamaged cars: any folder named like 'whole', 'undamaged', 'no_damage', or a Stanford Cars set.
UNDAMAGED_HINTS = ("01-whole", "whole", "undamaged", "no_damage", "not_damaged", "cars_train", "cars_test", "stanford")
vehide_roots = {split[k][1] for k in split}
undamaged_dirs = []
for dp, dns, fs in os.walk(INPUT_ROOT):
    base = os.path.basename(dp).lower()
    if any(h in base for h in UNDAMAGED_HINTS) and not any(dp.startswith(r) for r in vehide_roots):
        if not any(dp.startswith(u + os.sep) for u in undamaged_dirs):
            undamaged_dirs.append(dp)
n_und = sum(len([f for f in fs if f.lower().endswith(IMG_EXTS)]) for u in undamaged_dirs for _, _, fs in os.walk(u))
print("Undamaged folders:", undamaged_dirs, "→", n_und, "images")
assert n_und > 0, "No undamaged-car images found — attach e.g. vinayjose/car-damage-dataset."

# Whole-car photos of DAMAGED cars (e.g. car-damage-dataset's 00-damage folders).
# VehiDE is mostly close-ups; without these the classifier learns
# "close-up = damaged, whole car = undamaged" instead of looking at the damage.
DAMAGED_NAMES = {"00-damage", "damage", "damaged", "damaged_cars", "crashed"}
damaged_dirs = []
for dp, dns, fs in os.walk(INPUT_ROOT):
    if os.path.basename(dp).lower() in DAMAGED_NAMES and not any(dp.startswith(r) for r in vehide_roots):
        if not any(dp.startswith(u + os.sep) for u in damaged_dirs):
            damaged_dirs.append(dp)
n_dmg = sum(len([f for f in fs if f.lower().endswith(IMG_EXTS)]) for u in damaged_dirs for _, _, fs in os.walk(u))
print("Damaged whole-car folders:", damaged_dirs, "→", n_dmg, "images")
if n_dmg == 0:
    print("WARNING: no whole-car damaged photos found; the classifier may judge framing instead of damage.")

In [ ]:
import yaml
prof = {"fast": dict(yolo_epochs=60, yolo_imgsz=640, yolo_batch=16),
        "full": dict(yolo_epochs=150, yolo_imgsz=1280, yolo_batch=8)}[PROFILE]

cfg = yaml.safe_load(open("config.yaml.example"))
cfg["paths"].update({
    "img_dir": split["train"][1],
    "json_path": split["train"][0],
    "val_img_dir": split.get("val", (None, None))[1],
    "val_json_path": split.get("val", (None, None))[0],
    "undamaged_dirs": undamaged_dirs,
    "damaged_dirs": damaged_dirs,
    "yolo_dataset": f"{WORK}/yolo_dataset",
    "data": f"{WORK}/yolo_dataset/data.yaml",
    "runs_segment": f"{WORK}/runs/segment",
    "output_model": f"{WORK}/export/resnet50_damage.pth",
    "resnet": f"{WORK}/export/resnet50_damage.pth",
    "yolo": f"{WORK}/export/yolo26_damage.pt",
})
cfg["models"]["yolo26m_pt"] = YOLO_START
cfg["training"].update(prof, device="auto", resnet_epochs=10, resnet_batch=64, workers=4)
yaml.safe_dump(cfg, open("config.yaml", "w"), sort_keys=False)
os.makedirs(f"{WORK}/export", exist_ok=True)
print(open("config.yaml").read())

## 2. Convert VehiDE polygons to a YOLO segmentation dataset

In [ ]:
!python -m app.models.YOLO26m.data_converter
!cat {WORK}/yolo_dataset/data.yaml

## 3. Train the ResNet-50 damaged / undamaged classifier (~10–20 min on a T4)

In [ ]:
!python -m app.models.Resnet50.resnet50

## 4. Train the YOLO26 segmentation model

If the session stops, re-run this cell: `yolo26m.py` resumes from `runs/segment/26m_pro/weights/last.pt`
(on Kaggle, save the notebook version with *Save & Run All* or copy `runs/` to a dataset first so it survives).

In [ ]:
!python -m app.models.YOLO26m.yolo26m
import shutil
shutil.copy(f"{WORK}/runs/segment/26m_pro/weights/best.pt", f"{WORK}/export/yolo26_damage.pt")

## 5. Evaluate both models on VehiDE's validation split

In [ ]:
imgsz = prof["yolo_imgsz"]
!python evaluate.py --imgsz {imgsz} --workers 4
import shutil, glob
shutil.copy("results/metrics.md", f"{WORK}/export/metrics.md")
shutil.copy("results/metrics.json", f"{WORK}/export/metrics.json")
for p in glob.glob("results/yolo_val/*.png"):
    shutil.copy(p, f"{WORK}/export/")
from IPython.display import Markdown
Markdown(open("results/metrics.md").read())

## 6. Quick end-to-end check on a few validation images

In [ ]:
import random, os
from app.damage_pipeline import analyze
val_dir = cfg["paths"]["val_img_dir"] or cfg["paths"]["img_dir"]
samples = random.sample([f for f in os.listdir(val_dir) if f.lower().endswith(IMG_EXTS)], 3)
und = [os.path.join(dp, f) for u in undamaged_dirs for dp, _, fs in os.walk(u) for f in fs if f.lower().endswith(IMG_EXTS)]
for path in [os.path.join(val_dir, s) for s in samples] + random.sample(und, 2):
    seg, cls, dmg, _ = analyze(path)
    print(os.path.basename(path), "|", cls, "|", dmg.replace("\n", "  "))

## 7. Download

Grab everything in `/kaggle/working/export/` (Output tab), then in your local repo:

```bash
mkdir -p Backend/weights
cp resnet50_damage.pth yolo26_damage.pt Backend/weights/
```

Paste `metrics.md` into the README's *Performance & Results* section.